In [1]:
import torch
import torch.nn.functional as F
from torch.distributions import Categorical
import os

from pilgrim import PermutationEnv
from pilgrim import PilgrimPolicy
from pilgrim import PilgrimValue
from pilgrim import PPOTrainer


In [2]:
# Пример использования
policy_net = PilgrimPolicy(
    state_size=10,
    n_actions=2,
    hd1=500,
    hd2=100,
    nrd=2,
    dropout_rate=0.0,
    activation_function="relu",
    use_batch_norm=False
)
value_net  = PilgrimValue(
    state_size=10,
    hd1=500,
    hd2=100,
    nrd=2,
    output_dim=1,
    dropout_rate=0.0,
    activation_function="relu",
    use_batch_norm=False
)

In [5]:
trainer = PPOTrainer(
    policy_net,
    value_net,
    n=10,
    device='cuda',
    max_steps_start=1,
    max_steps_final=10,
    curriculum_steps=20,
    mini_batch_size = 1024,
    rollout_size = 10
)

In [6]:
# Запустить
trainer.run(total_iterations=100, save_interval=10)

iteration 0
curriculum 0.0


c:\Users\zamko\Documents\pilgrim\pilgrim\trainer_ppo.py:120: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  state = torch.tensor(self.env.reset(random_start=True), dtype=torch.float32, device=self.device)
c:\Users\zamko\Documents\pilgrim\pilgrim\trainer_ppo.py:148: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  state = torch.tensor(next_s, dtype=torch.float32, device=self.device)
c:\Users\zamko\Documents\pilgrim\pilgrim\trainer_ppo.py:153: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  state = torch.tensor(self.env.reset(random_start=True), 

collect_trajectories 0.2522578239440918
compute_advantages 0.0009999275207519531
loss tensor(0.1512, device='cuda:0', grad_fn=<SubBackward0>)
update 0.182023286819458
iteration 1
curriculum 0.0


c:\Users\zamko\Documents\pilgrim\pilgrim\trainer_ppo.py:242: UserWarning: Using a target size (torch.Size([10, 1])) that is different to the input size (torch.Size([10])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  value_loss = F.mse_loss(value_pred, ret_b)


collect_trajectories 0.03000473976135254
compute_advantages 0.0010008811950683594
loss tensor(-0.0444, device='cuda:0', grad_fn=<SubBackward0>)
update 0.0815134048461914
iteration 2
curriculum 0.0
collect_trajectories 0.028000354766845703
compute_advantages 0.0010001659393310547
loss tensor(0.0091, device='cuda:0', grad_fn=<SubBackward0>)
update 0.07851195335388184
iteration 3
curriculum 0.0
collect_trajectories 0.028999805450439453
compute_advantages 0.0010004043579101562
loss tensor(-0.0136, device='cuda:0', grad_fn=<SubBackward0>)
update 0.07905459403991699
iteration 4
curriculum 0.0
collect_trajectories 0.031000852584838867
compute_advantages 0.0009992122650146484
loss tensor(-0.0135, device='cuda:0', grad_fn=<SubBackward0>)
update 0.07851481437683105
iteration 5
curriculum 0.0
collect_trajectories 0.029000043869018555
compute_advantages 0.0010006427764892578
loss tensor(0.0115, device='cuda:0', grad_fn=<SubBackward0>)
update 0.07851052284240723
iteration 6
curriculum 0.0
collect_t

In [7]:
import torch
from torch.distributions import Categorical
import random

def test_policy(policy_net, env, k=10, max_steps=200, device='cpu'):
    """
    Пример теста политики:
      1) Берём тождественную перестановку [0,1,2,...,n-1].
      2) Совершаем k случайных шагов в среде, получаем "перемешанное" состояние.
      3) Запускаем policy_net на этом состоянии, пытаемся вернуться к тождественному.
      4) Возвращаем tuple (done, steps_used).
         done=True, если достигли тождественной перестановки за <= max_steps.
    """
    policy_net.eval()

    # 1) Сбрасываем среду в тождественную (random_start=False, если вы так сделали)
    #    Предположим, что в env.reset(...) можно передать random_start=False
    #    и оно вернёт [0,1,2,...].
    state = env.reset(random_start=False)

    # 2) "Закрутим" k раз случайно
    #    Здесь мы вручную делаем k шагов с выбором случайного допустимого действия.
    for _ in range(k):
        # В env может быть несколько действий (например, 2). Выбираем случайное.
        # Либо, если в env есть метод do_random_step, используем его.
        action = random.randrange(2)  # Или len(env.possible_actions)
        next_s, _, done, _ = env.step(action)
        state = next_s
        if done:
            # Если вдруг случайно уже вернулись в тождественное — "перемешивание" не особо удалось :)
            break

    # Теперь state — "запутанное" состояние.
    # 3) Запускаем policy, пытаемся вернуться к цели
    steps_used = 0
    done = False

    # Превращаем state в тензор
    state_t = torch.tensor(state, dtype=torch.float32, device=device)
    print('state_t', state_t)

    for _ in range(max_steps):
        steps_used += 1

        # Получаем logits из policy
        logits = policy_net(state_t.unsqueeze(0))  # [1, n_actions]
        dist = Categorical(logits=logits)
        action = dist.sample()  # [1]
        
        # Делаем шаг в среде
        next_s, reward, done, info = env.step(action.item())
        print('next_s', next_s)
        
        # Проверяем, не достигли ли цели
        if done:
            return True, steps_used
        
        # Обновляем state_t
        state_t = torch.tensor(next_s, dtype=torch.float32, device=device)

    # Если за max_steps не дошли до цели, возвращаем неудачу
    return False, steps_used

def test_policy_multiple_runs(
    policy_net,
    env,
    num_runs=10,
    random_steps=10,
    max_steps=200,
    device='cpu'
):
    """
    Запускаем test_policy несколько раз, чтобы посчитать статистику успеха.
    """
    successes = 0
    total_steps = 0

    for i in range(num_runs):
        done, steps_used = test_policy(
            policy_net, env,
            k=random_steps,
            max_steps=max_steps,
            device=device
        )
        total_steps += steps_used
        if done:
            successes += 1

    success_rate = successes / num_runs
    avg_steps = total_steps / num_runs
    print(f"Success rate: {success_rate*100:.1f}% ({successes}/{num_runs})")
    print(f"Average steps used (including failures): {avg_steps:.1f}")

    return success_rate, avg_steps


In [8]:
trainer.env.max_steps = 300

In [12]:
done, steps_used = test_policy(trainer.policy_net, trainer.env, k=5, max_steps=300, device='cuda')
print(done, steps_used)


state_t tensor([0., 1., 2., 3., 4., 5., 6., 7., 8., 9.], device='cuda:0')
next_s tensor([1, 0, 2, 3, 4, 5, 6, 7, 8, 9], device='cuda:0')
next_s tensor([0, 1, 2, 3, 4, 5, 6, 7, 8, 9], device='cuda:0')
True 2


C:\Users\zamko\AppData\Local\Temp\ipykernel_18624\345625208.py:39: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  state_t = torch.tensor(state, dtype=torch.float32, device=device)
C:\Users\zamko\AppData\Local\Temp\ipykernel_18624\345625208.py:59: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  state_t = torch.tensor(next_s, dtype=torch.float32, device=device)
